# Legal-BERT Fine-Tuning for NyayaSetu

This notebook allows you to fine-tune a pre-trained Legal-BERT model on your own custom legal dataset (e.g., `extracted_text.txt`).

### Steps:
1. Upload your `extracted_text.txt` file to the Colab Files section (folder icon on the left).
2. Run the cells below in order.
3. Download the trained model when finished.

In [ ]:
# 1. Install Required Libraries
!pip install transformers datasets accelerate torch

In [ ]:
import os
# Disable W&B logging to avoid interactive prompts
os.environ["WANDB_DISABLED"] = "true"

from transformers import (
    BertTokenizer,
    BertForMaskedLM,
    LineByLineTextDataset,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

# Check if GPU is available
import torch
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: You are using CPU. This will be slow. Go to Runtime > Change runtime type > T4 GPU.")

In [ ]:
# 2. Configuration
# We use 'nlpaueb/legal-bert-base-uncased' which is already pre-trained on legal text.
base_model = 'nlpaueb/legal-bert-base-uncased'
dataset_file = 'extracted_text.txt'  # Make sure you uploaded this!
output_dir = './nyayasetu-legal-bert'

# Check if file exists
if not os.path.exists(dataset_file):
    raise FileNotFoundError(f"Please upload '{dataset_file}' to the Colab files area before continuing.")

In [ ]:
# 3. Load Model and Tokenizer
print("Loading tokenizer and model...")
tokenizer = BertTokenizer.from_pretrained(base_model)
model = BertForMaskedLM.from_pretrained(base_model)

In [ ]:
# 4. Prepare Dataset
print("Tokenizing dataset... this may take a moment.")
dataset = LineByLineTextDataset(
    tokenizer=tokenizer,
    file_path=dataset_file,
    block_size=128,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15
)

In [ ]:
# 5. Training Settings
training_args = TrainingArguments(
    output_dir=output_dir,
    overwrite_output_dir=True,
    num_train_epochs=5,           # Increase this for better results (e.g., 10 or 20)
    per_device_train_batch_size=8,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    learning_rate=2e-5,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

In [ ]:
# 6. Start Training
print("Starting training...")
trainer.train()

In [ ]:
# 7. Save the Model
print("Saving model...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

In [ ]:
# 8. Test the Model
from transformers import pipeline

fill_mask = pipeline(
    "fill-mask",
    model=output_dir,
    tokenizer=output_dir
)

# Test basic legal context
test_sentence = "The plaintiff filed a [MASK] against the company."
results = fill_mask(test_sentence)

print(f"Test Sentence: {test_sentence}")
for p in results:
    print(f"{p['token_str']}: {p['score']:.4f}")

In [ ]:
# 9. Zip and Download
import shutil
from google.colab import files

print("Zipping model for download...")
shutil.make_archive("nyayasetu_model", 'zip', output_dir)

files.download("nyayasetu_model.zip")